<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Read distributions

**[Read distributions](https://learning.nextia-ai.com/courses/math/m01/read-distributions/)** · Essential Mathematics and Statistics for AI · Module 1, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** count a histogram of 1,200 reply times with different bin widths, find the outliers with the 1.5 × IQR rule, see what a right skew does to the mean, and do the module practice: two teams with the same mean and a different spread.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | Nothing to install and no account: only Python's standard library. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | The mean, the median, the quartiles, the IQR and the SD of Monday's 12 tickets, from [Describe a dataset: Percentiles](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/#percentiles-where-a-value-stands) and [Spread](https://learning.nextia-ai.com/courses/math/m01/describe-a-dataset/#spread-range-variance-and-standard-deviation). |
| **Tested** | Python 3.14.6 (standard library only), in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m01/read-distributions/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M01-L03-read-distributions/read-distributions-solution.ipynb).

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

## Setup: the week of reply times

The next cell defines `week`: the first-reply times, in minutes, of all 1,200 tickets of one week at Larkfield, as on the lesson page. The last 36 values are the tickets from Monday's outage. The data is fictional.

Run the cell. You should see `Ready: 1200 tickets, 32043 minutes in total`.

In [ ]:
# The data, written in full (you do not need to read it).
week = [
    21, 15, 28, 18, 13, 40, 42, 35, 8, 42, 24, 23, 17, 17, 17, 12, 19, 39, 30, 18,
    26, 34, 40, 37, 39, 17, 48, 30, 22, 13, 30, 19, 35, 13, 10, 5, 33, 21, 25, 33,
    34, 26, 12, 56, 26, 22, 14, 12, 30, 22, 23, 19, 26, 23, 27, 29, 44, 22, 27, 50,
    19, 63, 25, 27, 22, 11, 11, 38, 25, 24, 19, 11, 22, 29, 33, 24, 25, 18, 31, 25,
    42, 34, 21, 40, 17, 21, 35, 45, 17, 14, 26, 40, 23, 67, 29, 28, 14, 11, 15, 12,
    27, 9, 38, 34, 26, 18, 24, 17, 19, 32, 39, 24, 12, 63, 15, 17, 33, 22, 37, 37,
    22, 13, 12, 12, 34, 19, 36, 25, 25, 19, 16, 33, 23, 8, 16, 25, 19, 10, 27, 31,
    27, 14, 21, 40, 14, 28, 21, 19, 13, 20, 20, 26, 11, 7, 30, 13, 11, 15, 32, 52,
    37, 32, 16, 20, 20, 21, 15, 13, 15, 30, 26, 31, 22, 35, 19, 34, 24, 23, 17, 21,
    21, 13, 18, 15, 16, 15, 18, 45, 29, 27, 42, 34, 17, 32, 17, 16, 13, 18, 44, 16,
    26, 20, 33, 48, 41, 20, 28, 19, 17, 22, 32, 13, 19, 17, 18, 22, 25, 26, 35, 16,
    15, 27, 34, 40, 13, 24, 21, 36, 17, 36, 24, 20, 25, 32, 18, 16, 13, 20, 24, 39,
    22, 14, 12, 13, 25, 19, 16, 17, 19, 30, 27, 21, 24, 14, 22, 13, 25, 25, 47, 18,
    30, 37, 34, 15, 21, 25, 27, 27, 18, 14, 12, 20, 27, 21, 32, 14, 14, 13, 19, 29,
    30, 20, 16, 30, 34, 19, 16, 12, 12, 35, 18, 14, 31, 43, 32, 20, 15, 13, 50, 25,
    30, 30, 50, 33, 33, 38, 26, 37, 24, 17, 13, 22, 23, 30, 48, 16, 29, 11, 18, 37,
    24, 8, 30, 39, 32, 21, 16, 31, 22, 34, 30, 18, 19, 20, 33, 25, 15, 36, 18, 17,
    16, 18, 19, 17, 15, 21, 21, 39, 30, 15, 36, 11, 15, 29, 21, 15, 72, 27, 26, 20,
    20, 32, 11, 24, 21, 50, 28, 20, 41, 30, 19, 26, 25, 32, 27, 79, 55, 23, 17, 27,
    22, 27, 13, 26, 17, 21, 21, 34, 41, 17, 22, 44, 20, 22, 30, 24, 12, 8, 16, 11,
    9, 16, 24, 23, 29, 29, 36, 24, 38, 19, 28, 22, 33, 25, 33, 13, 12, 19, 17, 30,
    10, 66, 16, 32, 22, 20, 26, 16, 29, 16, 26, 23, 9, 41, 15, 24, 12, 27, 17, 37,
    18, 25, 33, 20, 20, 35, 12, 17, 11, 11, 40, 23, 11, 20, 18, 48, 23, 16, 10, 31,
    29, 11, 22, 42, 19, 18, 23, 24, 18, 24, 78, 22, 30, 17, 12, 23, 19, 28, 28, 18,
    19, 10, 53, 10, 19, 23, 38, 28, 18, 28, 15, 27, 6, 20, 28, 18, 12, 32, 13, 38,
    19, 38, 14, 17, 27, 23, 24, 22, 23, 26, 21, 18, 19, 17, 18, 27, 19, 27, 35, 25,
    23, 26, 9, 21, 48, 34, 24, 47, 9, 20, 20, 22, 24, 29, 28, 23, 13, 14, 19, 42,
    20, 30, 40, 14, 15, 10, 18, 28, 19, 20, 36, 29, 26, 15, 20, 25, 18, 28, 14, 40,
    17, 9, 41, 13, 40, 18, 49, 44, 35, 19, 33, 17, 19, 18, 31, 27, 13, 37, 27, 24,
    28, 22, 41, 20, 39, 22, 18, 28, 31, 19, 22, 19, 22, 14, 14, 24, 23, 45, 11, 39,
    26, 29, 12, 10, 42, 15, 19, 7, 14, 16, 24, 16, 21, 27, 39, 58, 33, 15, 34, 19,
    18, 14, 17, 26, 21, 16, 51, 15, 18, 17, 21, 35, 17, 18, 22, 33, 35, 19, 40, 12,
    7, 17, 38, 25, 29, 36, 45, 22, 48, 25, 11, 21, 13, 13, 10, 22, 30, 19, 10, 16,
    25, 26, 16, 48, 26, 20, 16, 31, 17, 17, 10, 25, 9, 24, 10, 18, 20, 23, 20, 21,
    18, 15, 20, 57, 19, 11, 10, 26, 42, 18, 14, 28, 9, 10, 22, 17, 19, 33, 14, 47,
    34, 21, 37, 21, 15, 27, 18, 24, 11, 9, 44, 20, 25, 29, 16, 10, 14, 26, 33, 15,
    31, 21, 21, 9, 16, 40, 25, 24, 14, 22, 16, 37, 26, 11, 26, 29, 61, 23, 26, 11,
    9, 27, 27, 21, 19, 23, 24, 27, 21, 21, 9, 32, 36, 18, 21, 17, 15, 16, 14, 7,
    14, 23, 26, 20, 11, 26, 45, 28, 19, 28, 18, 35, 27, 18, 18, 19, 23, 15, 22, 32,
    27, 19, 39, 12, 30, 17, 42, 27, 26, 31, 39, 17, 35, 33, 24, 26, 27, 45, 28, 56,
    25, 37, 30, 21, 14, 17, 45, 20, 13, 41, 39, 11, 16, 24, 39, 20, 15, 7, 19, 12,
    28, 14, 36, 56, 9, 21, 33, 18, 17, 23, 25, 16, 14, 21, 13, 17, 18, 23, 20, 13,
    30, 20, 42, 30, 35, 15, 19, 39, 21, 32, 17, 24, 36, 33, 16, 21, 36, 20, 19, 17,
    20, 40, 20, 17, 42, 29, 17, 19, 36, 16, 37, 34, 63, 29, 28, 9, 19, 22, 27, 17,
    19, 46, 19, 25, 11, 32, 7, 22, 10, 19, 33, 19, 12, 18, 11, 16, 10, 12, 12, 10,
    22, 20, 24, 43, 25, 17, 18, 22, 19, 25, 10, 15, 18, 33, 42, 35, 34, 24, 15, 18,
    28, 12, 48, 14, 45, 25, 14, 25, 18, 23, 41, 31, 26, 30, 15, 19, 32, 25, 22, 17,
    29, 19, 19, 21, 86, 26, 20, 17, 9, 44, 22, 11, 28, 22, 18, 15, 12, 38, 18, 18,
    28, 21, 33, 23, 15, 14, 6, 18, 21, 19, 49, 31, 26, 49, 23, 15, 34, 19, 28, 13,
    29, 37, 24, 34, 15, 25, 35, 34, 35, 30, 14, 13, 34, 22, 34, 11, 29, 43, 39, 21,
    17, 62, 20, 23, 22, 23, 14, 15, 9, 18, 14, 17, 28, 22, 20, 18, 23, 69, 42, 18,
    21, 18, 29, 38, 8, 9, 26, 62, 21, 15, 9, 26, 27, 31, 36, 33, 18, 27, 11, 18,
    32, 17, 43, 20, 14, 19, 15, 30, 27, 32, 27, 24, 27, 23, 27, 41, 26, 17, 14, 22,
    22, 25, 24, 22, 39, 21, 25, 44, 30, 20, 19, 20, 19, 79, 44, 44, 32, 69, 26, 41,
    24, 14, 27, 13, 37, 39, 27, 37, 21, 31, 14, 14, 21, 23, 5, 24, 21, 7, 29, 8,
    12, 17, 19, 12, 19, 28, 27, 19, 16, 21, 30, 24, 32, 42, 30, 21, 31, 32, 17, 49,
    14, 26, 26, 21, 52, 19, 28, 25, 17, 25, 24, 45, 11, 18, 60, 54, 18, 19, 16, 13,
    16, 47, 30, 27, 31, 28, 23, 23, 22, 23, 27, 20, 27, 25, 54, 19, 16, 19, 36, 26,
    20, 50, 13, 9, 93, 100, 107, 93, 98, 103, 96, 109, 94, 110, 98, 105, 107, 93, 101, 98,
    95, 109, 103, 106, 92, 93, 93, 94, 106, 108, 97, 91, 96, 99, 94, 99, 94, 106, 105, 96,
]
print(f"Ready: {len(week)} tickets, {sum(week)} minutes in total")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One dataset, three views

A **histogram** puts the values into **bins**: ranges of equal width. Each bin includes its left edge and not its right edge, so "10–20" means 10 to under 20 minutes. The next cell defines a function that counts the values in each bin, and draws Monday's 12 tickets with 10-minute bins.

> **Predict.** How many of Monday's 12 tickets are in the bin from 10 to under 20 minutes? Then run the cell.

In [ ]:
def histogram(values, width, start=0, end=120):
    """Count the values in each bin [left, left + width)."""
    lefts = list(range(start, end, width))
    counts = [len([v for v in values if left <= v < left + width]) for left in lefts]
    return lefts, counts

monday = [12, 15, 18, 18, 20, 22, 24, 25, 27, 31, 38, 95]
lefts, counts = histogram(monday, 10, end=100)
bars([f"{left}–{left + 10}" for left in lefts], counts, width=20)

> **Check.** You should see 4 tickets at 10–20, 5 at 20–30, 2 at 30–40, empty bins from 40 to 90, and 1 ticket at 90–100: T-2052.

A **box plot** draws the quartiles. Its **whiskers** reach the furthest values that are not outliers. A common rule calls a value an outlier if it is more than 1.5 × IQR beyond the box. Run the cell. You should see the limit `43.0` and the outliers `[95]`.

In [ ]:
q1, median, q3 = statistics.quantiles(monday, n=4, method="inclusive")
limit = q3 + 1.5 * (q3 - q1)
print("Q1", q1, " median", median, " Q3", q3)
print("upper limit", limit)
print("outliers", [v for v in monday if v > limit])

## 2. A full week: change the bin width

On the lesson page, the explorer changes the bin width of the week's 1,200 tickets. Here you do the same with the `histogram` function.

> **Predict.** The 36 outage tickets waited 90 to 110 minutes. With 10-minute bins, can you see them as a separate group? Then run the cell.

In [ ]:
lefts, counts = histogram(week, 10)
bars([f"{left}–{left + 10}" for left in lefts], counts)
print("total", sum(counts))

> **Check.** You should see the counts 36, 414, 411, 194, 76, 17, 11, 4, 1, 21, 14 and 1, and a total of 1200. After almost empty bins at 70 to 90 minutes, the bars at 90 to 110 minutes (21 and 14) rise again: the outage group.

> **Your turn: change one thing.** In the cell above, change the bin width from `10` to `40` and run it again. You should see three bars: 1055, 108 and 37. The outage group disappears into the last bar. Then try `20` and `5`.

With 1-minute bins, many bins are empty. Run the cell. You should see `37 of 120 bins are empty`.

In [ ]:
lefts, counts = histogram(week, 1)
print(counts.count(0), "of", len(counts), "bins are empty")

## 3. Skew: the mean and the median

The week is **right-skewed**: a long tail of slow tickets goes to the right. In a right-skewed distribution, the mean is larger than the median.

> **Predict.** How many of the 1,200 tickets waited less than the mean? More or less than half? Then run the cell.

In [ ]:
mean = statistics.mean(week)
print("mean  ", round(mean, 2))
print("median", statistics.median(week))
print("p90   ", statistics.quantiles(week, n=10, method="inclusive")[8])
below = len([v for v in week if v < mean])
print(f"below the mean: {below} of {len(week)} ({below / len(week):.1%})")

> **Check.** You should see a mean of 26.7, a median of 22.5, a p90 of 42.0, and 762 of 1200 (63.5%) below the mean. Nearly two in three customers waited less than the "average" wait.

## 4. Outliers: find the cause before you decide

Apply the 1.5 × IQR rule to the week. Run the cell. You should see Q1 17.0, Q3 31.0, the limit 52.0 and 61 flagged tickets, of which 36 are outage tickets.

In [ ]:
q1, median, q3 = statistics.quantiles(week, n=4, method="inclusive")
limit = q3 + 1.5 * (q3 - q1)
flagged = [v for v in week if v > limit]
outage = week[-36:]                       # the last 36 values are the outage tickets
print("Q1", q1, " Q3", q3, " limit", limit)
print("flagged:", len(flagged))
print("of them from the outage:", len([v for v in flagged if v >= 90]), "| ordinary slow tickets:", len([v for v in flagged if v < 90]))
print("outage tickets waited", min(outage), "to", max(outage), "minutes")

> **Check.** 36 of the 61 flagged tickets are from the outage; the other 25 are ordinary slow tickets in the long tail. The rule cannot tell the two kinds apart. Only a person who checks the cause can.

## 5. Worked example: Grace's weekly report

Priya adds the values that the mean hides. Run the cell. You should see 1200 tickets, a median of 22.5, a p90 of 42.0, 132 tickets (11.0%) over 40 minutes, 36 outage tickets and a mean of 26.7.

In [ ]:
over_40 = len([v for v in week if v > 40])
table(["Value", "Result"], [
    ["Tickets", len(week)],
    ["Median (minutes)", float(statistics.median(week))],
    ["p90 (minutes)", statistics.quantiles(week, n=10, method="inclusive")[8]],
    ["Waited more than 40 minutes", f"{over_40} ({over_40 / len(week):.1%})"],
    ["Outage group", len(outage)],
    ["Mean (minutes)", round(statistics.mean(week), 1)],
], digits=1)

## 6. Guided practice: read a histogram from its counts

These are the counts of the week's histogram with 10-minute bins, as on the lesson page.

> **Your turn.** Use only `counts_10` (not `week`). Replace each `...` with a calculation, then run the check cell.
>
> - `total`: the sum of all counts.
> - `under_40`: the tickets under 40 minutes (the first four bins), and `share_under_40` as a percentage.
> - `from_60`: the tickets that waited 60 minutes or more.
> - `median_bin`: the left edge of the bin where the running total first passes 600.

In [ ]:
counts_10 = [36, 414, 411, 194, 76, 17, 11, 4, 1, 21, 14, 1]
lefts_10 = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110]

total = sum(counts_10)
under_40 = sum(counts_10[:4])
share_under_40 = under_40 / total * 100
from_60 = sum(counts_10[6:])
running = 0
for left, n in zip(lefts_10, counts_10):
    running += n
    if running > 600:
        median_bin = left
        break

Run the check cell.

In [ ]:
expect("the total", total, 1200)
expect("the number of tickets under 40 minutes", under_40, 1055)
expect("the share under 40 minutes", share_under_40, 87.9, tol=0.05)
expect("the number of tickets at 60 minutes or more", from_60, 52)
expect("the left edge of the median bin", median_bin, 20)

The median bin, 20–30 minutes, agrees with the median of 22.5 minutes.

## 7. Independent variation: without the outage

Mei removes the 36 outage tickets.

> **Predict.** Does the mean go up or down? Does it change more or less than the median? Which bars change?

> **Your turn.** Make the list without the outage (the outage tickets are the last 36 values of `week`). Calculate its mean and its median. Then run the check cell.

In [ ]:
without_outage = week[:-36]
mean_without = statistics.mean(without_outage)
median_without = statistics.median(without_outage)

Run the check cell.

In [ ]:
expect("the mean without the outage", mean_without, 24.45)
expect("the median without the outage", median_without, 22)

The mean falls by about 2.25 minutes, the median by only 0.5. Run the cell to see the new histogram: the bars from 90 to 120 minutes are empty, and all other bars are the same. The long tail up to 86 minutes is still there, so the data is still right-skewed.

In [ ]:
lefts, counts = histogram(week[:-36], 10)
bars([f"{left}–{left + 10}" for left in lefts], counts)

## 8. Failure case: deleting outliers without asking why

Tomás's data had every value beyond 1.5 × IQR deleted, because "outliers are noise". The next cell does the same to the week. Run it.

> **Predict.** How many tickets does the rule delete, and how many of them are ordinary slow tickets, not the outage?

In [ ]:
kept = [v for v in week if v <= 52]
deleted = [v for v in week if v > 52]
print("deleted:", len(deleted), "| ordinary slow tickets among them:", len([v for v in deleted if v < 90]))
print("mean before", round(statistics.mean(week), 2), "| mean after", round(statistics.mean(kept), 2))

> **Check.** You should see 61 deleted tickets, of which 25 are ordinary slow tickets, and the mean falls from 26.7 to 23.59. A model trained on `kept` never sees a slow, complex ticket. **Fix:** find the cause of each group of outliers first, remove only wrong or out-of-scope values, and write down the rule.

## 9. Module practice: same mean, different spread

### Worked example: two groups of three

Agent X answers in 9, 10 and 11 minutes, agent Y in 0, 10 and 20. Run the cell. You should see the same mean, 10, an SD of 0.82 for X and 8.16 for Y, and ranges of 2 and 20.

In [ ]:
for agent, values in [("X", [9, 10, 11]), ("Y", [0, 10, 20])]:
    mean = statistics.mean(values)
    squares = [(v - mean) ** 2 for v in values]
    print(agent, "mean", mean, "| squared distances", squares, "| sum", sum(squares),
          "| SD", round(statistics.pstdev(values), 2), "| range", max(values) - min(values))

### Your task: returns and delivery

Grace compares eight tickets from each team. Larkfield promises a first reply within 40 minutes.

> **Your turn.** For each team, calculate the mean, the median, the SD (divide by n), the range and the number of tickets over 40 minutes. Then run the check cell. Write your two or three sentences for Grace in `c03-notes.md`.

In [ ]:
returns = [22, 23, 24, 25, 25, 26, 27, 28]
delivery = [8, 12, 18, 25, 25, 32, 38, 42]

mean_returns, mean_delivery = statistics.mean(returns), statistics.mean(delivery)
median_returns, median_delivery = statistics.median(returns), statistics.median(delivery)
sd_returns, sd_delivery = statistics.pstdev(returns), statistics.pstdev(delivery)
range_returns, range_delivery = max(returns) - min(returns), max(delivery) - min(delivery)
over_40_returns = len([v for v in returns if v > 40])
over_40_delivery = len([v for v in delivery if v > 40])

Run the check cell.

In [ ]:
expect("the mean of returns", mean_returns, 25)
expect("the mean of delivery", mean_delivery, 25)
expect("the median of returns", median_returns, 25)
expect("the median of delivery", median_delivery, 25)
expect("the SD of returns", sd_returns, 1.87)
expect("the SD of delivery", sd_delivery, 11.26)
expect("the range of returns", range_returns, 6)
expect("the range of delivery", range_delivery, 34)
expect("the number of returns tickets over 40 minutes", over_40_returns, 0)
expect("the number of delivery tickets over 40 minutes", over_40_delivery, 1)

Both teams have a mean and a median of 25 minutes, but the delivery team's SD is about 6 times as large, and one of its tickets broke the 40-minute promise. The model answer for the sentences is on the lesson page.

## Check your understanding and recap

Answer the **knowledge checks** and the **module check** on the [lesson page](https://learning.nextia-ai.com/courses/math/m01/read-distributions/#module-check). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you counted the week's histogram with different bin widths, and saw that 10-minute bins show the outage group and 40-minute bins hide it. The week is right-skewed: the mean (26.7 minutes) is above the wait of 63.5% of customers. The 1.5 × IQR rule flags 61 tickets, but only 36 are from the outage. Two teams with the same mean can have very different spreads.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Represent observations](https://learning.nextia-ai.com/courses/math/m02/represent-observations/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m01/read-distributions/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.